# Current multi-commodity benchmark generation and validation

This notebook uses the current canonical generator and writes 12 F1?F4 scenarios ? 10 seeds = **120 instances** by default. The filenames retain `v3` for existing links.

Both generation notebooks import `generate_dissertation_scenarios.py`. General commodities are `G1`?`G4` in the multi-commodity notebook, with the additional special product `S1`; the single-commodity notebook uses `G1` with an empty special-product set and unrestricted routing. Modes are `F`, `P30`, and `P70`.

**Restart the kernel, then Run All.** Review the settings cell first. Generation overwrites matching JSON files and reports in the selected output folder. Keep the two notebooks' seeds, horizon, and physical parameters matched.

Structural validation and regression tests do not certify MILP feasibility. Exact direct-MILP solves are available near the end and are off by default. No solver results are invented or inferred from generator diagnostics.

## 1. Load the current implementation

In [1]:
from pathlib import Path
import hashlib
import importlib
import io
import json
import sys
import unittest
from collections import defaultdict

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in (cwd, *cwd.parents)
     if (path / "model_data_loader.py").is_file()
     and (path / "synthetic_data_generator" / "generate_dissertation_scenarios.py").is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the repository or its synthetic_data_generator folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Refuse a different interpreter/package set before generating or solving.
from scripts.check_environment import check_environment
environment_report = check_environment()
if not environment_report["passed"]:
    raise RuntimeError(
        "Select MATH4003 requirements (Python 3.9.6, Gurobi 12.0.3) from .venv-requirements. "
        + "; ".join(environment_report["errors"])
    )


import model_data_loader
from synthetic_data_generator import generate_dissertation_scenarios as gen
importlib.reload(model_data_loader)
importlib.reload(gen)

print("Python:", sys.version.split()[0])
print("Generator:", gen.__file__)
print("Generator version:", gen.GENERATOR_RANDOMIZATION_VERSION)
print("RNG stream version:", gen.RNG_STREAM_VERSION)
assert set(gen.CANONICAL_MODEL_READY_KEYS) == set(model_data_loader.CANONICAL_MODEL_READY_KEYS)

Python: 3.12.10
Generator: C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\generate_dissertation_scenarios.py
Generator version: frozen_schema_routing_restricted_v2
RNG stream version: matched_single_multi_streams_v1


## 2. Settings

In [2]:
COMMODITY_COUNT = 4  # General commodities only; 4 also adds S1.
SEEDS = tuple(range(1, 11))
SCENARIO_NAMES = tuple(gen.SCENARIO_NAMES)
HORIZON_DAYS = 90
OUTPUT_DIR = PROJECT_ROOT / "synthetic_data_generator" / "multicommodity_scenarios"

# Use the current calibrated physical defaults in both notebooks.
CONFIG = gen.GeneratorConfig(horizon_days=HORIZON_DAYS, commodity_count=COMMODITY_COUNT)
CONFIG.validate()
assert SEEDS and len(SEEDS) == len(set(SEEDS))
assert SCENARIO_NAMES and len(SCENARIO_NAMES) == len(set(SCENARIO_NAMES))

# Optional direct-MILP feasibility/optimisation check after generation.
RUN_EXACT_MILP = False
MAX_EXACT_SCENARIOS = None  # Set 1 for a smoke test; None tests every generated instance.
TIME_LIMIT_SECONDS = 600
THREADS_PER_SOLVE = 1
STRICT_CAPACITY = False  # Penalised overflow; also the variant used by Benders runners.

print("Output:", OUTPUT_DIR)
print("Planned instances:", len(SCENARIO_NAMES) * len(SEEDS))
print("Configuration:", CONFIG)

Output: C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\multicommodity_scenarios
Planned instances: 120
Configuration: GeneratorConfig(horizon_days=90, seed=1, commodity_count=4, operating_hours_per_day=24.0, train_unloader_rate_tph=2900.0, stacker_rate_tph=2300.0, reclaimer_rate_tph=2700.0, shiploader_rate_tph=4600.0, secondary_arc_rate_fraction=0.8)


## 3. Preview the current schema and routing

In [3]:
preview = gen.build_scenario(SCENARIO_NAMES[0], CONFIG)
preview_data = preview.optimizer_objects()
preview.validate()
print("Scenario:", preview.scenario_name)
print("General commodities:", preview_data["G"])
print("Special products:", preview_data["SPECIAL_PRODUCTS"])
print("Modes:", preview_data["K"])
print("Assets / arcs / maintenance:", len(preview_data["A"]), len(preview_data["E"]), len(preview_data["M"]))
print("Initial inventory entries:", len(preview_data["initial_inventory_tonnes"]))
for product in preview_data["SPECIAL_PRODUCTS"]:
    allowed_zones = [zone for zone in preview_data["P"]
                     if preview_data["zone_allowed_product"][(zone, product)]]
    route = [(preview_data["arc_from"][arc], preview_data["arc_to"][arc])
             for arc in preview_data["E"] if preview_data["product_arc_compatible"][(product, arc)]]
    print(product, "allowed zones:", allowed_zones)
    print(product, "allowed route:", route)

Scenario: F1_system_scale_L
General commodities: ['G1', 'G2', 'G3', 'G4']
Special products: ['S1']
Modes: ['F', 'P30', 'P70']
Assets / arcs / maintenance: 12 34 30
Initial inventory entries: 20
S1 allowed zones: ['Z4']
S1 allowed route: [('RAIL_SOURCE', 'TU2'), ('TU2', 'ST3'), ('ST3', 'Z4'), ('Z4', 'RC4'), ('RC4', 'SL2'), ('SL2', 'VESSEL_SINK')]


## 4. Generate all selected instances

The generator writes `benchmark_manifest.json`, `benchmark_validation.csv`, `family_validation.csv`, and `seed_diversity.csv`. The family CSV is a descriptive summary, not an independent pass/fail certificate. Generation calls the canonical structural validator for every scenario.

In [4]:
generated = gen.generate_benchmark_suite(
    OUTPUT_DIR, config=CONFIG, scenario_names=SCENARIO_NAMES, seeds=SEEDS,
)
manifest_path = generated["benchmark_manifest"]
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
expected_count = len(SCENARIO_NAMES) * len(SEEDS)
assert manifest["instance_count"] == manifest["expected_instance_count"] == expected_count
assert manifest["commodity_count"] == COMMODITY_COUNT
print("Generated:", manifest["instance_count"])
for name in ("benchmark_manifest.json", "benchmark_validation.csv", "family_validation.csv", "seed_diversity.csv"):
    print(" -", OUTPUT_DIR / name)

Generated: 120
 - C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\multicommodity_scenarios\benchmark_manifest.json
 - C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\multicommodity_scenarios\benchmark_validation.csv
 - C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\multicommodity_scenarios\family_validation.csv
 - C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\multicommodity_scenarios\seed_diversity.csv


## 5. Re-load every exported file and verify routing, hashes, and seed diversity

Validation uses the manifest's exact file list. Unexpected instance JSON files in the output folder are rejected so later recursive solver runners do not mix old and new instances. File hashes record this generation; they do not certify solver feasibility.

In [5]:
instance_paths = [(OUTPUT_DIR / entry["file"]).resolve() for entry in manifest["files"]]
assert len(instance_paths) == len(set(instance_paths)) == expected_count
actual_paths = {path.resolve() for path in OUTPUT_DIR.glob("*/seed_*/*.json")}
assert actual_paths == set(instance_paths), "Unexpected/missing instances: use a clean output folder or review stale files"
expected_pairs = {(name, seed) for name in SCENARIO_NAMES for seed in SEEDS}
assert {(entry["scenario_name"], entry["seed"]) for entry in manifest["files"]} == expected_pairs

file_checks = []
fingerprints = defaultdict(set)
for entry, path in zip(manifest["files"], instance_paths):
    loaded = model_data_loader.load_canonical_scenario(path, require_single_commodity=(COMMODITY_COUNT == 1))
    data = loaded["model_ready_data"]
    assert data["G"] == [f"G{i}" for i in range(1, COMMODITY_COUNT + 1)]
    assert data["SPECIAL_PRODUCTS"] == (["S1"] if COMMODITY_COUNT == 4 else [])
    assert data["K"] == ["F", "P30", "P70"]
    assert data["D"] == list(range(HORIZON_DAYS))
    if COMMODITY_COUNT == 4:
        assert [zone for zone in data["P"] if data["zone_allowed_product"][f"{zone}::S1"]] == ["Z4"]
        assert all(data["zone_allowed_product"][f"Z4::{product}"] == 0 for product in data["G"])
        route = {(i, j) for i, j in loaded["E"] if loaded["arc_ok"](i, j, "S1")}
        node, visited, used = "RAIL_SOURCE", set(), set()
        while node != "VESSEL_SINK":
            assert node not in visited, "Cycle in S1 route"
            visited.add(node)
            outgoing = [(i, j) for i, j in route if i == node]
            assert len(outgoing) == 1, "S1 must have one complete dedicated route"
            used.add(outgoing[0])
            node = outgoing[0][1]
        assert "Z4" in visited and used == route
    fingerprint = hashlib.sha256(json.dumps(data, sort_keys=True).encode("utf-8")).hexdigest()
    fingerprints[entry["scenario_name"]].add(fingerprint)
    file_checks.append({
        "file": entry["file"], "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
        "structural_pass": True,
    })

assert all(len(fingerprints[name]) == len(SEEDS) for name in SCENARIO_NAMES), "Duplicate seed realizations"
print("Exported instances validated:", len(file_checks))
print("Distinct realizations per scenario:", {name: len(values) for name, values in fingerprints.items()})

Exported instances validated: 120
Distinct realizations per scenario: {'F1_system_scale_L': 10, 'F1_system_scale_M': 10, 'F1_system_scale_H': 10, 'F2_mode_tradeoff_L': 10, 'F2_mode_tradeoff_M': 10, 'F2_mode_tradeoff_H': 10, 'F3_yard_flow_L': 10, 'F3_yard_flow_M': 10, 'F3_yard_flow_H': 10, 'F4_operational_conflict_L': 10, 'F4_operational_conflict_M': 10, 'F4_operational_conflict_H': 10}


## 6. Run current generator regressions and save the validation record

These tests cover the implemented F1 scale ladder, F2 mode tradeoffs, F3 yard stress, F4 activity conflicts, matched physical single/multi data, deterministic seeds, and special-product routing. They replace the obsolete supply/demand, old asset-count, and protected-campaign checks. They test generator behavior on the cases selected by the test suite; the preceding cell separately validates every exported instance.

In [6]:
test_module = importlib.import_module("synthetic_data_generator.test_generate_dissertation_scenarios")
importlib.reload(test_module)
suite = unittest.defaultTestLoader.loadTestsFromModule(test_module)
stream = io.StringIO()
result = unittest.TextTestRunner(stream=stream, verbosity=1).run(suite)
print(stream.getvalue())
assert result.wasSuccessful() and not result.skipped, "Generator regression tests did not all pass"

validation_record = {
    "generator_randomization_version": gen.GENERATOR_RANDOMIZATION_VERSION,
    "commodity_count": COMMODITY_COUNT,
    "instance_count": len(file_checks),
    "all_structural_checks_pass": True,
    "all_seed_diversity_checks_pass": True,
    "generator_regression_tests_run": result.testsRun,
    "generator_regression_tests_pass": True,
    "solver_certified": False,
    "files": file_checks,
}
validation_path = OUTPUT_DIR / "notebook_validation.json"
validation_path.write_text(json.dumps(validation_record, indent=2) + "\n", encoding="utf-8")
print("Wrote:", validation_path)
print("Data checks passed. Exact MILP feasibility has not been established by these checks.")

.................
----------------------------------------------------------------------
Ran 17 tests in 0.588s

OK

Wrote: C:\Ponpot_Work_Space\math4003-port-yard-optimisation\synthetic_data_generator\multicommodity_scenarios\notebook_validation.json
Data checks passed. Exact MILP feasibility has not been established by these checks.


## 7. Optional exact direct-MILP run

Enable `RUN_EXACT_MILP` in the settings cell to run the actual current solver on the generated files. An incumbent establishes feasibility; proven `INFEASIBLE` establishes infeasibility. A time limit with no incumbent, a license error, or another unresolved status does not establish infeasibility. A feasible incumbent alone does not establish optimality. The CSV records time limits, threads, solver status, gap, and bounds.

In [7]:
if RUN_EXACT_MILP:
    from scenario_runner import run_scenarios, write_csv, print_results
    solver_module = importlib.import_module("monolithic_milp" if COMMODITY_COUNT == 1 else "multicommodity_milp")
    solver_module = importlib.reload(solver_module)
    paths = instance_paths if MAX_EXACT_SCENARIOS is None else instance_paths[:MAX_EXACT_SCENARIOS]
    exact_results = run_scenarios(
        paths=[str(path) for path in paths],
        model_label="single_commodity" if COMMODITY_COUNT == 1 else "multi_commodity",
        load_scenario=solver_module.load_scenario,
        build_model=solver_module.build_model,
        strict_capacity=STRICT_CAPACITY,
        time_limit_seconds=TIME_LIMIT_SECONDS,
        threads_per_solve=THREADS_PER_SOLVE,
        workers=1,
        show_progress=True,
    )
    print_results(exact_results)
    exact_csv = OUTPUT_DIR / "exact_milp_results.csv"
    write_csv(exact_results, str(exact_csv))
    classifications = defaultdict(int)
    for row in exact_results:
        if (row.get("solution_count") or 0) > 0:
            classification = "FEASIBLE"
        elif row.get("status") == "INFEASIBLE":
            classification = "INFEASIBLE"
        else:
            classification = "UNRESOLVED"
        classifications[classification] += 1
    print("Feasibility classifications:", dict(classifications))
    print("Wrote:", exact_csv)
    print("All generated instances shown feasible:",
          len(exact_results) == expected_count and classifications["FEASIBLE"] == expected_count)
else:
    print("Exact MILP run is off. Set RUN_EXACT_MILP = True to enable it.")

Exact MILP run is off. Set RUN_EXACT_MILP = True to enable it.


## 8. Run all direct MILPs and Benders comparisons

After generating both datasets, restart the kernel in `../run_model_tests.ipynb`. Use `MODEL = "both"`, `SCENARIO_INPUTS = None`, `MAX_SCENARIOS = 1` for a smoke test, then `MAX_SCENARIOS = None` for all instances. It reads the default output folders used here. If you change `OUTPUT_DIR`, set corresponding inputs in that notebook.

From a terminal in the repository root, run routing/loader regressions:

```powershell
python -m unittest test_milp_canonical_loaders synthetic_data_generator.test_generate_dissertation_scenarios -v
```

Benders comparisons run both direct MILP and Benders for each instance. First smoke-test:

```powershell
python run_benders_scenarios.py --limit 1 --max-iters 200 --tol 0.0001
python run_benders_scenarios_multicommodity.py --limit 1 --max-iters 200 --tol 0.0001
```

Then replace `--limit 1` with `--all` to run every instance. These scripts read the default `singlecommodity_scenarios` and `multicommodity_scenarios` folders and use penalised overflow. They have no time-limit option. Treat the printed comparisons as correctness diagnostics, not a controlled timing comparison with time-limited notebook runs. `MAX_ITERS` is unresolved convergence; inspect `OPTIMAL` statuses and objective agreement within tolerance. Solver-related skipped tests do not count as solver verification.

The former `base_control` and `structurally_infeasible` scenario names are not supported by the current generator and are not generated here. Do not infer all benchmark instances are feasible from structural checks or replace difficult seeds based on solver performance.